# 03 - Preprocessing

Split first, then impute/scale/engineer features — fit only on train, to avoid any leakage into test.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
import joblib

DATA_PATH = Path("../data/raw/diabetes.csv")
PROCESSED_PATH = Path("../data/processed")
MODEL_PATH = Path("../outputs/models")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()


**Step 1: Treat disguised missing values (0 -> NaN)**

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)
print(df.isnull().sum())


**Step 2: Split BEFORE any fitting (prevents leakage)**

In [ ]:
TARGET = "Outcome"
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)


**Step 3: KNN Imputation — fit on train only**

In [ ]:
imputer = KNNImputer(n_neighbors=5)

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

print("Missing after imputation - train:", X_train.isnull().sum().sum(), " test:", X_test.isnull().sum().sum())


**Step 4: Feature engineering**

Adding domain-informed interaction and binned features. Fit strictly on train statistics where applicable (bin edges here are clinical thresholds, not data-derived, so no leakage).

In [ ]:
def add_features(df):
    df = df.copy()
    df["Glucose_BMI"] = df["Glucose"] * df["BMI"]
    df["Age_Glucose"] = df["Age"] * df["Glucose"]
    df["BMI_Age"] = df["BMI"] * df["Age"]
    df["Glucose_bin"] = pd.cut(df["Glucose"], [0, 100, 126, 300], labels=[0, 1, 2]).astype(int)
    df["BMI_bin"] = pd.cut(df["BMI"], [0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)
    df["Insulin_Glucose_ratio"] = df["Insulin"] / (df["Glucose"] + 1)
    return df

X_train = add_features(X_train)
X_test = add_features(X_test)

print(X_train.shape, X_test.shape)
X_train.head()


**Step 5: Scale — fit on train only**

In [ ]:
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Scaling completed successfully.")


**Step 6: Save processed data + fitted transformers**

In [ ]:
train = X_train_scaled.copy()
train["Outcome"] = y_train.reset_index(drop=True)
test = X_test_scaled.copy()
test["Outcome"] = y_test.reset_index(drop=True)

train.to_csv(PROCESSED_PATH / "train_processed.csv", index=False)
test.to_csv(PROCESSED_PATH / "test_processed.csv", index=False)

joblib.dump(imputer, MODEL_PATH / "knn_imputer.pkl")
joblib.dump(scaler, MODEL_PATH / "robust_scaler.pkl")

print("Processed datasets and transformers saved successfully.")
print("Train:", train.shape, " Test:", test.shape)
